In [15]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple

# =========================
# Settings
# =========================

# Feed characteristics (µm midpoints)
ultra_fine_sizing = 50       # ultrafines class midpoint
fine_sizing = 112.5          # fines class midpoint
coarse_sizing = 175          # coarse class midpoint

DEFAULT_SETTINGS = {
	"site": {"name": "Demo CIL/Leach", "timezone": "Africa/Dar_es_Salaam"},
	"tanks": {
		"leach": {"count": 2, "volume_m3": 1000.0},     # per tank
		"cil":   {"count": 9, "volume_m3": 250.0},       # per tank
	},
	"hydraulics": {
		"slurry_density_t_per_m3": 1.60,
		"target_pH_leach": 10.6,
		"target_pH_cil": 10.5,
	},
	"ore": {
		"d_threshold_um": 20.0,                  # characteristic size (µm)
		"locked_gold_threshold": 0.02,           # 2% locked gold
		"liberation_fraction": 1.0 - 0.02,       # derived (can override explicitly)
		"density_t_per_m3": 2.7,                 # ore density (t/m3)
		"liquid_density_t_per_m3": 1.15          # liquid density (t/m3)
	},
	"grind": {
		# Average particle sizes for each class (µm). Adjusting these affects kinetics & recovery.
		"particle_sizes_um": {
			"Coarse": 175.0,
			"Fines": 112.5,
			"Ultrafines": 50.0
		}
	},
	"carbon": {
		"cil_carbon_residence_days_per_tank": 3.0,
		"loading_floor_gt": 450.0,               # last tank min
		"loading_cap_gt": 1300.0,                # first tank cap
		"barren_loading_gt": 60.0
	},
	"recovery": {
		"floor_pct": 80.0,                       # lower bound for day recovery
		"max_efficiency": 0.96,                  # recovery cap multiplier on liberation
		# weights combine to form a 0..1 score (we renormalize internally)
		"weights": {"cn": 0.55, "grade": 0.25, "do": 0.15, "grind": 0.05, "ph": 0.00},
		"noise_sd": 0.5                          # %-points
	},
	"kinetics": {
		"k_diss_base": 0.9,
		"k_ads_base":  1.1
	},
	"losses": {
		"system_loss_frac_min": 0.003,
		"system_loss_frac_max": 0.012,
		"inventory_sigma_frac": 0.003,
		"residual_dissolved_outflow_frac_max": 0.003
	},
	"outliers": {
		"upset_day_rate": 0.008,
		"late_tank_bias_low": 0.10,
		"late_tank_bias_high": 0.25
	},
	"ag_factor_range": (1.9, 2.4)               # Ag = Au * U[1.9, 2.4]
}

# =========================
# Quantile-mapping infra
# =========================
@dataclass
class FieldSpec:
	qmin: float; q25: float; q50: float; q75: float; qmax: float
	seasonality_amp: float = 0.0
	seasonality_period: int = 365
	smooth_noise_sd: float = 0.5
	rho: float = 0.85
	outlier_rate: float = 0.0
	outlier_mult_range: Tuple[float, float] = (1.05, 1.25)
	clip_min: float | None = None
	clip_max: float | None = None

def smooth_base_series(n, seed, amp=0.0, period=365, noise_sd=0.5, rho=0.85):
	rng = np.random.default_rng(seed)
	t = np.arange(n)
	seasonal = amp * np.sin(2*np.pi * t / max(period, 2))
	eps = rng.normal(0, noise_sd, n)
	x = np.zeros(n)
	for i in range(1, n):
		x[i] = rho * x[i-1] + eps[i]
	return seasonal + x

def map_to_quantiles(base_values, spec: FieldSpec, seed):
	rng = np.random.default_rng(seed)
	n = len(base_values)
	order = np.argsort(base_values)
	ranks = np.empty(n, dtype=float)
	ranks[order] = np.linspace(0, 1, n, endpoint=True)
	ranks = np.clip(ranks + rng.normal(0, 0.002, n), 0, 1)
	ps = np.array([0.00, 0.25, 0.50, 0.75, 1.00])
	vs = np.array([spec.qmin, spec.q25, spec.q50, spec.q75, spec.qmax], dtype=float)
	mapped = np.interp(ranks, ps, vs)
	if spec.outlier_rate > 0:
		mask = rng.random(n) < spec.outlier_rate
		mult = rng.uniform(spec.outlier_mult_range[0], spec.outlier_mult_range[1], mask.sum())
		mapped[mask] = np.minimum(mapped[mask] * mult, spec.qmax)
	if spec.clip_min is not None or spec.clip_max is not None:
		lo = spec.clip_min if spec.clip_min is not None else -np.inf
		hi = spec.clip_max if spec.clip_max is not None else np.inf
		mapped = np.clip(mapped, lo, hi)
	return mapped

def robust_unit(x):
	lo, hi = np.quantile(x, [0.05, 0.95])
	if hi - lo < 1e-9:
		return np.zeros_like(x)
	z = (x - lo) / (hi - lo)
	return np.clip(z, 0.0, 1.0)

# =========================
# Specs from real-world stats
# =========================
SPECS = {
	"Leach_Feed_Throughput_M3Hr": FieldSpec(
		82.25, 138.62, 142.90, 148.30, 290.86,
		seasonality_amp=3.0, smooth_noise_sd=2.5, outlier_rate=0.012, outlier_mult_range=(1.2, 1.9),
		clip_min=110, clip_max=200
	),
	"Percent_Solids": FieldSpec(
		31.00, 49.29, 50.38, 51.13, 61.00,
		seasonality_amp=0.6, smooth_noise_sd=0.6, outlier_rate=0.004, outlier_mult_range=(1.05, 1.15),
		clip_min=48.5, clip_max=53.5
	),
	"Leach_Feed_Grade_Au_Gt_Ds": FieldSpec(
		1.01, 1.83, 2.15, 2.44, 4.13, seasonality_amp=0.10, smooth_noise_sd=0.22, clip_min=1.0, clip_max=3.8
	),
	"Leach_Feed_Grade_Au_Gt_Ns": FieldSpec(
		0.89, 1.81, 2.16, 2.45, 3.64, seasonality_amp=0.10, smooth_noise_sd=0.20, clip_min=0.9, clip_max=3.6
	),
	"Leach_Feed_Grade_Au_Gt_Day": FieldSpec(
		1.13, 1.85, 2.15, 2.45, 3.51, seasonality_amp=0.06, smooth_noise_sd=0.18, clip_min=1.0, clip_max=3.6
	),
	"WAD_CN_Tailings_ppm": FieldSpec(
		30.0, 200.0, 268.0, 345.0, 940.0, smooth_noise_sd=0.2, outlier_rate=0.01,
		outlier_mult_range=(1.2, 2.0), clip_min=50, clip_max=500
	),
	# Size splits (as %). We'll rename outputs to Coarse/Fines/Ultrafines
	"Coarse": FieldSpec(1.51, 6.85, 8.75, 12.35, 22.57, smooth_noise_sd=0.25, clip_min=2.0, clip_max=22.6),
	"Fines":  FieldSpec(4.86, 15.15, 17.68, 19.96, 29.19, smooth_noise_sd=0.25, clip_min=10.0, clip_max=29.2),
	"Ultrafines": FieldSpec(57.56, 68.07, 73.97, 77.71, 84.94, smooth_noise_sd=0.25, clip_min=55.0, clip_max=85.0),
	"Leach_Feed_Dry_t": FieldSpec(
		643.89, 2346.58, 2525.21, 2560.29, 2705.59,
		smooth_noise_sd=30.0, outlier_rate=0.003, outlier_mult_range=(0.6, 1.1),
		clip_min=1800, clip_max=2800
	),
}

# =========================
# Helper functions
# =========================

# --- Convert dry tonnes/day to slurry m3/day using densities & %solids (w/w) ---
def tons_per_day_to_slurry_m3_per_day(dry_tpd, percent_solids_wt, ore_density_tpm3, liquid_density_tpm3):
	"""
	dry_tpd: dry solids mass (t/day)
	percent_solids_wt: % solids by weight (0-100)
	ore_density_tpm3: solids density (t/m3)
	liquid_density_tpm3: liquid density (t/m3)
	Returns slurry volumetric throughput (m3/day)
	Logic: Cw = Ms/(Ms+Mw). Given Ms and Cw -> Mw = Ms * (1-Cw)/Cw
		   V = Ms/rho_s + Mw/rho_l
	"""
	Ms = np.asarray(dry_tpd, dtype=float)                  # t/day (solids)
	Cw = np.clip(np.asarray(percent_solids_wt, float)/100, 1e-6, 0.999999)  # w/w
	Mw = Ms * (1.0 - Cw) / Cw                               # t/day (liquid)
	V_s = Ms / max(ore_density_tpm3, 1e-6)                  # m3/day
	V_l = Mw / max(liquid_density_tpm3, 1e-6)               # m3/day
	return V_s + V_l

def calculate_p80(row: pd.Series, coarse_col: str = "coarse",
                  fines_col: str = "fines", ultrafines_col: str = "ultrafines") -> float:
    """
    Estimate P80 (µm) by linear interpolation across cumulative size bins.
    Expects percentage columns for ultrafines, fines, coarse that sum ~100.
    """
    sizes = [50, 112.5, 175]
    fractions = [row[ultrafines_col], row[fines_col], row[coarse_col]]

    cumulative = np.cumsum(fractions)  # % passing

    for i in range(1, len(cumulative)):
        if cumulative[i - 1] <= 80 <= cumulative[i]:
            x0, x1 = cumulative[i - 1], cumulative[i]
            y0, y1 = sizes[i - 1], sizes[i]
            return y0 + (80 - x0) * (y1 - y0) / (x1 - x0)

    return np.nan  # 80% not reached


def calculate_particle_size(ultrafine_pct: float, fine_pct: float, coarse_pct: float,
                            size_ultrafine: float = ultra_fine_sizing,
                            size_fine: float = fine_sizing,
                            size_coarse: float = coarse_sizing) -> float:
    """Weighted-average particle size (µm) from % by class."""
    return (
        ultrafine_pct * size_ultrafine +
        fine_pct * size_fine +
        coarse_pct * size_coarse
    ) / 100.0

def ph_series(n, seed_base, offset):
    return smooth_base_series(n, seed_base, noise_sd=0.08, rho=0.9) + offset

# =========================
# Main generator (updated)
# =========================
def generate_demo_historical_full(
	start_date="2024-01-01",
	end_date="2025-12-31",
	seed=2025,
	settings: Dict = None
):
	S = settings or DEFAULT_SETTINGS
	rng = np.random.default_rng(seed)

	# Dates
	dates = pd.date_range(start_date, end_date, freq="D")
	n = len(dates)

	# --- % Solids FIRST (so it's available for m3/day conversion) ---
	perc_solids = map_to_quantiles(
		smooth_base_series(
			n, seed+2,
			amp=SPECS["Percent_Solids"].seasonality_amp,
			period=SPECS["Percent_Solids"].seasonality_period,
			noise_sd=SPECS["Percent_Solids"].smooth_noise_sd,
			rho=SPECS["Percent_Solids"].rho
		),
		SPECS["Percent_Solids"], seed+12
	)

	# --- Anchor on Leach_Feed_Dry_t (dry t/day) ---
	base_dry = smooth_base_series(n, seed+19, amp=2.0, period=365, noise_sd=30.0, rho=0.92)
	daily_tons = map_to_quantiles(base_dry, SPECS["Leach_Feed_Dry_t"], seed+119)

	# --- Convert dry t/day + %solids + densities -> slurry m3/day ---
	ore_rho = S["ore"]["density_t_per_m3"]
	liq_rho = S["ore"]["liquid_density_t_per_m3"]
	m3day = tons_per_day_to_slurry_m3_per_day(
		dry_tpd=daily_tons,
		percent_solids_wt=perc_solids,
		ore_density_tpm3=ore_rho,
		liquid_density_tpm3=liq_rho
	)

	# --- Tight m3/hr nameplate (140–150); runtime is the flex ---
	rng_local = np.random.default_rng(seed+911)
	m3hr_set = np.clip(145.0 + rng_local.normal(0, 1.2, n), 140.0, 150.0)

	runtime_hours = m3day / m3hr_set
	needs_over_24 = runtime_hours > 24.0
	if np.any(needs_over_24):
		m3hr_needed = m3day[needs_over_24] / 24.0
		m3hr_set[needs_over_24] = np.clip(m3hr_needed, 140.0, 150.0)
		runtime_hours[needs_over_24] = m3day[needs_over_24] / m3hr_set[needs_over_24]

	# --- Grades (unchanged) ---
	base_au_ds = smooth_base_series(
		n, seed+3,
		amp=SPECS["Leach_Feed_Grade_Au_Gt_Ds"].seasonality_amp,
		period=SPECS["Leach_Feed_Grade_Au_Gt_Ds"].seasonality_period,
		noise_sd=SPECS["Leach_Feed_Grade_Au_Gt_Ds"].smooth_noise_sd,
		rho=SPECS["Leach_Feed_Grade_Au_Gt_Ds"].rho
	)
	au_ds = map_to_quantiles(base_au_ds, SPECS["Leach_Feed_Grade_Au_Gt_Ds"], seed+13)

	base_au_ns = smooth_base_series(
		n, seed+4,
		amp=SPECS["Leach_Feed_Grade_Au_Gt_Ns"].seasonality_amp,
		period=SPECS["Leach_Feed_Grade_Au_Gt_Ns"].seasonality_period,
		noise_sd=SPECS["Leach_Feed_Grade_Au_Gt_Ns"].smooth_noise_sd,
		rho=SPECS["Leach_Feed_Grade_Au_Gt_Ns"].rho
	)
	au_ns = map_to_quantiles(base_au_ns, SPECS["Leach_Feed_Grade_Au_Gt_Ns"], seed+14)
	au_day = map_to_quantiles(0.5*(au_ds + au_ns), SPECS["Leach_Feed_Grade_Au_Gt_Day"], seed+15)

	# Ag derived from Au (row-wise factor in [1.9, 2.4])
	ag_lo, ag_hi = S["ag_factor_range"]
	ag_ds = au_ds * rng.uniform(ag_lo, ag_hi, n)
	ag_ns = au_ns * rng.uniform(ag_lo, ag_hi, n)
	ag_day = 0.5*(ag_ds + ag_ns)

	# Size splits (as %) now named Coarse/Fines/Ultrafines
	# We keep them correlated to P80-like drivers but map to your targets
	p80_proxy = np.clip(70 + 4*np.sin(np.linspace(0, 2*np.pi, n)) + \
					 np.random.default_rng(seed+5).normal(0, 2.0, n), 58, 82)
	coarse = map_to_quantiles(p80_proxy + smooth_base_series(
						n, seed+6, amp=0.5, noise_sd=0.2, rho=0.9), SPECS["Coarse"], seed+61)
	fines  = map_to_quantiles(-p80_proxy + smooth_base_series(
						n, seed+7, amp=0.5, noise_sd=0.2, rho=0.9), SPECS["Fines"],  seed+62)
	ultraf = 100.0 - (coarse + fines)
	ultraf = np.clip(ultraf, SPECS["Ultrafines"].clip_min or \
				  -np.inf, SPECS["Ultrafines"].clip_max or np.inf)

	# Grind-reactivity index: higher if more fine surface area; responds to avg sizes in settings
	sizes = S["grind"]["particle_sizes_um"]
	size_map = {"Coarse": sizes["Coarse"], "Fines": sizes["Fines"], "Ultrafines": sizes["Ultrafines"]}
	# Surface index ~ sum(frac / size), normalized 0..1
	surface_index = (coarse/size_map["Coarse"] + fines/size_map["Fines"] + ultraf/size_map["Ultrafines"])
	grind_index = robust_unit(surface_index)

	# -----------------------------
	# pH (tight bands)
	# -----------------------------
	PH_SPEC_LEACH = FieldSpec(10.30, 11.21, 11.57, 11.82, 12.86,
							smooth_noise_sd=0.06, clip_min=10.2, clip_max=12.9)
	PH_SPEC_CIL   = FieldSpec(10.30, 11.34, 11.55, 11.74, 12.58,
							smooth_noise_sd=0.06, clip_min=10.2, clip_max=12.8)
	
	# Leach pH (kept as before but via shared helper)
	ph_leach_01 = map_to_quantiles(ph_series(n, seed+71, 11.55), PH_SPEC_LEACH, seed+171)
	ph_leach_02 = map_to_quantiles(ph_series(n, seed+72, 11.55), PH_SPEC_LEACH, seed+172)

	# NEW: pH for every CIL tank (clustered around target_pH_cil with small tank trend)
	target_pH_cil = S["hydraulics"]["target_pH_cil"]
	ncil = S["tanks"]["cil"]["count"]
	rng_pH_cil = np.random.default_rng(seed+173)
	cil_idx_01 = np.arange(1, ncil+1)[None, :]
	cil_pH_base = (target_pH_cil
				- 0.03 * (cil_idx_01 - 1)                          # slight downward drift down-circuit
				+ rng_pH_cil.normal(0, 0.04, (n, ncil)))           # small noise
	ph_cil_all = np.empty((n, ncil))
	for j in range(ncil):
		# map through spec for realistic spread (keeps within clip bounds)
		ph_cil_all[:, j] = map_to_quantiles(cil_pH_base[:, j], PH_SPEC_CIL, seed+174+j)

	# DO profiles
	do_leach_1 = np.clip(20.0 + np.random.default_rng(seed+200).normal(0, 0.6, n), 17.5, 22.0)
	do_leach_2 = np.clip(19.2 + np.random.default_rng(seed+201).normal(0, 0.6, n), 17.0, 21.5)
	do_mean_leach = 0.5*(do_leach_1 + do_leach_2)

	# CN (kg/t) driven by Au_day + throughput (as before)
	au_norm = robust_unit(au_day)
	th_norm = robust_unit(m3day)
	daily_cn_kgt = np.clip(0.9 + 0.25*au_norm + 0.05*th_norm + np.random.default_rng(seed+300).normal(0, 0.05, n), 0.6, 1.4)

	# Normalized CN index used for free-CN profiles
	CN_z = robust_unit(daily_cn_kgt)

	# MTD kg/t
	tmp = pd.DataFrame({"date": dates, "daily": daily_cn_kgt})
	tmp["YM"] = tmp["date"].dt.to_period("M")
	mtd_cn_kgt = (tmp.groupby("YM")["daily"].apply(lambda s: s.expanding(1).mean())
						 .reset_index(level=0, drop=True).to_numpy())

	# CN tailings ppm
	base_cn_tails = daily_cn_kgt + np.random.default_rng(seed+400).normal(0, 0.05, n)
	cn_tails_ppm = map_to_quantiles(base_cn_tails, SPECS["WAD_CN_Tailings_ppm"], seed+401)

	# Leach free CN: TK-01 high, TK-02 slightly higher
	base_cn = 180 + 200*CN_z   # 180–380 nominal before noise
	free_cn_tk1 = np.clip(base_cn + np.random.default_rng(seed+402).normal(0, 10, n), 200, 360)
	free_cn_tk2 = np.clip(free_cn_tk1 * (1.02 + \
							np.random.default_rng(seed+403).normal(0, 0.01, n)), 205, 380)

	# CIL free CN: start below TK-02 and decay monotonically
	rng_cil = np.random.default_rng(seed+404)
	ncil = S["tanks"]["cil"]["count"]
	cn_cil = np.zeros((n, ncil))
	cn_cil[:, 0] = np.clip(free_cn_tk2 * \
						rng_cil.uniform(0.85, 0.93, n) + rng_cil.normal(0, 4, n), 120, 340)
	for j in range(1, ncil):
		# geometric decay with small noise, bounded so it never increases
		next_val = cn_cil[:, j-1] * rng_cil.uniform(0.88, 0.96, n) + rng_cil.normal(0, 3, n)
		cn_cil[:, j] = np.clip(np.minimum(next_val, cn_cil[:, j-1] * 0.98), 60, 320)


	# CIL DO (decay)
	cil_idx = np.arange(S["tanks"]["cil"]["count"])[None, :]
	do_cil = np.clip(18.0 - 0.3*cil_idx + \
				np.random.default_rng(seed+405).normal(0, 0.4,
							(n, S["tanks"]["cil"]["count"])), 12.0, 19.5)

	# Carbon conc (g/L) — seed, re-used to scale adsorption later
	carbon_cil = np.clip(14.0 - 0.3*cil_idx + \
				np.random.default_rng(seed+406).normal(0, 0.6,
							(n, S["tanks"]["cil"]["count"])), 8.0, 20.0)

	# ====================================================
	# Recovery% with DO + grind influence and liberation cap
	# ====================================================
	weights = S["recovery"]["weights"].copy()
	# normalize weights to sum 1
	sw = sum(weights.values()) or 1.0
	for k in weights: weights[k] /= sw

	DO_z = robust_unit(do_mean_leach)

	# pH term: small bell around target
	target_pH = S["hydraulics"]["target_pH_leach"]
	ph_dev = np.abs(0.5*(ph_leach_01 + ph_leach_02) - target_pH)
	pH_term = 1.0 - np.clip(ph_dev/1.0, 0, 1)  # ~1 near target, fades as you move 1 pH unit away

	# score 0..1 from weighted factors
	score = (weights["cn"]*CN_z +
			 weights["grade"]*au_norm +
			 weights["do"]*DO_z +
			 weights["grind"]*grind_index +
			 weights.get("ph", 0.0)*pH_term)

	# Liberation cap
	lib_base = S["ore"]["liberation_fraction"]
	# dynamic: a touch higher with finer grind
	lib_dyn = np.clip(lib_base * (0.95 + 0.1*grind_index), 0.80*lib_base, 1.05*lib_base)
	recovery_cap = 100.0 * (S["recovery"]["max_efficiency"] * lib_dyn)  # e.g., ~92–96%

	# Map score to [floor, cap] with a gentle S-shape
	floor = S["recovery"]["floor_pct"]
	# logistic-ish transform for nonlinearity
	score_sig = 1/(1 + np.exp(-3*(score - 0.5)))
	raw_recovery = floor + (recovery_cap - floor) * score_sig
	recovery_pct = np.clip(raw_recovery + \
						np.random.default_rng(seed+500).normal(
								0, S["recovery"]["noise_sd"], n), floor, recovery_cap)

	Au_Feed_g = au_day * daily_tons
	Au_Recovered_g = Au_Feed_g * (recovery_pct / 100.0)
	Recovery_pct_out = recovery_pct

	# ========================================
	# Masses, losses, inventory & per-tank flows
	# ========================================
	Au_feed_g = au_day * daily_tons

	sys_loss_frac = np.random.default_rng(seed+600).\
				uniform(S["losses"]["system_loss_frac_min"], S["losses"]["system_loss_frac_max"], n)
	System_Losses_g = Au_feed_g * sys_loss_frac
	Delta_Inventory_g = np.random.default_rng(seed+601).\
				normal(0.0, S["losses"]["inventory_sigma_frac"]*Au_feed_g)
	Residual_Dissolved_Outflow_g = np.clip(
		(S["losses"]["residual_dissolved_outflow_frac_max"] * Au_feed_g) * (1 - DO_z),
		0, None
	)

	Au_tails_g = Au_feed_g - Au_Recovered_g - System_Losses_g - Delta_Inventory_g
	neg_mask = Au_tails_g < 0
	if neg_mask.any():
		take = np.minimum(-Au_tails_g[neg_mask], Residual_Dissolved_Outflow_g[neg_mask])
		Residual_Dissolved_Outflow_g[neg_mask] -= take
		Au_tails_g[neg_mask] += take
		Au_tails_g = np.clip(Au_tails_g, 0, None)

	Au_ads_target = np.clip(Au_Recovered_g + Delta_Inventory_g + \
						 System_Losses_g - Residual_Dissolved_Outflow_g, 0, None)

	# Operating intensity for stage split & kinetics
	O = 0.55 * CN_z + 0.30 * DO_z + 0.15 * grind_index
	P = 0.25 * robust_unit(ultraf)  # front-load penalty from ultrafines

	# Residence times
	Q_m3_per_day = m3day
	tau_leach = (S["tanks"]["leach"]["volume_m3"] * np.ones((n, 2))) / Q_m3_per_day[:, None]
	tau_cil = (S["tanks"]["cil"]["volume_m3"] * np.\
			ones((n, S["tanks"]["cil"]["count"]))) / Q_m3_per_day[:, None]
	
	# Weight fractions:
	Cw = np.clip(perc_solids/100.0, 1e-6, 0.999999)      # solids (w/w)
	Ms_tpd = daily_tons                                  # solids mass t/day
	Mw_tpd = Ms_tpd * (1.0 - Cw) / Cw                    # liquid mass t/day

	Solids_Fraction = Cw
	Liquid_Fraction = 1.0 - Cw

	# Masses (kg/day)
	Slurry_Mass_kg   = (Ms_tpd + Mw_tpd) * 1000.0
	Solution_Mass_kg = Mw_tpd * 1000.0

	# Flow volume (L/day)
	L_per_day = m3day * 1000.0

	# NaCN usage estimates (kg/day and kg/t)
	NaCN_Used_Est_kg_day = daily_cn_kgt * Ms_tpd                  # kg/t * t/day = kg/day
	NaCN_Used_Est_kgt    = daily_cn_kgt                           # passthrough alias

	# -----------------------------
	# Leach kinetics & front-loaded split (gold)
	# -----------------------------
	k_diss = S["kinetics"]["k_diss_base"] * (1 + 0.7 * CN_z + 0.4 * DO_z + 0.3 * grind_index) \
			* (S["ore"]["density_t_per_m3"]/S["ore"]["density_t_per_m3"])
	k_diss = np.clip(k_diss, 0.4, 2.5)

	# Residence times already computed (tau_leach)
	# Strongly front-load leach: majority in TK-01, the rest in TK-02
	base_f1 = 1 - np.exp(-k_diss * np.clip(tau_leach[:, 0], 0.05, 6.0))
	f1 = 0.70 + 0.18 * robust_unit(base_f1) + 0.12 * O - 0.18 * P  # 0.70–0.93 range
	f1 = np.clip(f1 + np.random.default_rng(seed+510).normal(0, 0.01, len(f1)), 0.70, 0.93)

	# Most dissolution is completed in leach: leave only a small remainder for CIL
	base_f2 = 1 - np.exp(-0.7 * k_diss * np.clip(tau_leach[:, 1], 0.05, 6.0))
	f_leach = 0.86 + 0.06 * robust_unit(base_f1 * 0.6 + base_f2 * 0.4) + 0.05 * O - 0.04 * P  # 0.86–0.96
	f_leach = np.clip(f_leach, 0.86, 0.96)

	Au_dissolved_total_needed = Au_ads_target + Residual_Dissolved_Outflow_g
	Au_diss_leach_total = f_leach * Au_dissolved_total_needed
	Au_diss_leach_1 = f1 * Au_diss_leach_total
	Au_diss_leach_2 = Au_diss_leach_total - Au_diss_leach_1

	# Consistent: solution at leach outlets (for plotting)
	Sol_Au_Leach_1_out = Au_diss_leach_1
	Sol_Au_Leach_2_out = Au_diss_leach_total

	# Remaining (small) dissolution allowed in CIL
	Au_diss_cil_total = Au_dissolved_total_needed - Au_diss_leach_total

	# Beta-shaped weights across CIL (fallback to geometric if SciPy missing)
	def beta_weights(n_tanks, a, b):
		i = np.arange(1, n_tanks+1)
		x = (i - 0.5) / n_tanks
		from scipy.stats import beta as beta_dist
		w = beta_dist.pdf(x, a, b)
		w = np.clip(w, 1e-6, None)
		return w / w.sum()

	try:
		_ = beta_weights(9, 2.0, 5.0)
		use_beta = True
	except Exception:
		use_beta = False

	if use_beta:
		a_gold = 1.2 + 4.0*O
		b_gold = 3.0 + 2.0 * P
		W_gold = np.vstack([beta_weights(S["tanks"]["cil"]["count"],
								   a_gold[i], b_gold[i]) for i in range(n)])
	else:
		r = np.clip(0.70 + 0.20 * O - 0.10 * P, 0.55, 0.90)
		W_gold = np.array([r[i]**np.arange(S["tanks"]["cil"]["count"]) for i in range(n)])
		W_gold = W_gold / W_gold.sum(axis=1, keepdims=True)

	Au_diss_cil_by_tank = Au_diss_cil_total[:, None] * W_gold

	# Adsorption per tank (stronger overall; extra in early CIL)
	k_ads0 = S["kinetics"]["k_ads_base"]
	ads_factor = (0.5 * robust_unit(carbon_cil) + 0.3 * robust_unit(do_cil) + 0.2 * robust_unit(cn_cil))
	k_ads_mat = k_ads0 * (1.0 + 1.6*ads_factor)  # stronger than before
	k_ads_mat[:, 0] *= 1.35                      # CIL-01 extra pull
	k_ads_mat[:, 1] *= 1.12                      # CIL-02 slight extra

	gold_diss_after_leach = Au_diss_leach_1 + Au_diss_leach_2
	Diss_Au_CIL = np.zeros((n, S["tanks"]["cil"]["count"]))
	ads_gold_per_tank = np.zeros_like(Diss_Au_CIL)

	# Target per-tank maximum outlet ratio (strict decay caps)
	decay_caps = np.array([0.88, 0.92, 0.94, 0.95, 0.965, 0.975, 0.98, 0.985, 0.99])

	flow = gold_diss_after_leach.copy()  # previous outlet (starts at end-of-leach)
	for j in range(S["tanks"]["cil"]["count"]):
		add = Au_diss_cil_total * 0.0    # << optional: zero extra dissolution in CIL for a clean decay
		# If you prefer a tiny residual dissolution, use:
		# add = Au_diss_cil_by_tank[:, j] * 0.35  # damp the residual CIL dissolution strongly

		c_in = flow + add
		base_out = c_in * np.exp(-np.clip(k_ads_mat[:, j], 0.3, 4.0) * np.clip(tau_cil[:, j], 0.1, 4.0))

		# Enforce CIL-j outlet <= previous outlet * decay cap (strictly non-increasing)
		out_cap = flow * decay_caps[j]
		out = np.minimum(base_out, out_cap)

		ads = np.maximum(c_in - out, 0.0)
		Diss_Au_CIL[:, j] = out
		ads_gold_per_tank[:, j] = ads
		flow = out

	# Ensure CIL-01 is below end-of-leach by a healthy margin
	Diss_Au_CIL[:, 0] = np.minimum(Diss_Au_CIL[:, 0],
						Sol_Au_Leach_2_out * np.random.default_rng(seed+512).uniform(0.78, 0.90, n))

	# Rescale adsorption to hit Au_ads_target exactly (row-wise)
	ads_total = ads_gold_per_tank.sum(axis=1)
	scale = np.clip(np.divide(Au_ads_target, np.maximum(ads_total, 1e-9)), 0.7, 1.4)
	ads_gold_per_tank *= scale[:, None]
	Diss_Au_CIL *= scale[:, None]  # scale solution consistently to keep mass balance with the simple loop definition
	# Silver dissolved (slower adsorption)
	Ag_feed_g = ag_day * daily_tons
	Ag_recovered_g = Ag_feed_g * np.clip(0.65 + 0.12 * au_norm + \
						np.random.default_rng(seed + 700).normal(0, 0.01, n), 0.55, 0.88)
	Ag_ads_target = Ag_recovered_g
	Ag_diss_leach_total = Ag_ads_target * (0.45 + 0.20 * O - 0.10 * P)
	Ag_diss_leach_1 = Ag_diss_leach_total * (0.55 + 0.15 * O)
	Ag_diss_leach_2 = Ag_diss_leach_total - Ag_diss_leach_1
	Sol_Ag_Leach_1_out = Ag_diss_leach_1
	Sol_Ag_Leach_2_out = Ag_diss_leach_1 + Ag_diss_leach_2

	Ag_diss_cil_total = Ag_ads_target - Ag_diss_leach_total
	
	if use_beta:
		a_s = 1.1 + 2.5 * O; b_s = 3.2 + 1.8 * P
		W_s = np.vstack([beta_weights(S["tanks"]["cil"]["count"],
								a_s[i], b_s[i]) for i in range(n)])
	else:
		r_s = np.clip(0.75 + 0.15 * O - 0.05 * P, 0.60, 0.92)
		W_s = np.array([r_s[i]**np.arange(S["tanks"]["cil"]["count"]) for i in range(n)])
		W_s = W_s / W_s.sum(axis=1, keepdims=True)
	Ag_diss_cil_by_tank = Ag_diss_cil_total[:, None] * W_s
	k_ads_mat_ag = 0.6 * k_ads_mat
	Diss_Ag_CIL = np.zeros_like(Diss_Au_CIL)
	ads_silver_per_tank = np.zeros_like(Diss_Au_CIL)
	flow = (Ag_diss_leach_1 + Ag_diss_leach_2).copy()
	for j in range(S["tanks"]["cil"]["count"]):
		add = Ag_diss_cil_by_tank[:, j]
		c_in = flow + add
		out = c_in * np.exp(-np.clip(k_ads_mat_ag[:, j], 0.15, 3.0) * np.clip(tau_cil[:, j], 0.1, 4.0))
		ads = np.maximum(c_in - out, 0.0)
		Diss_Ag_CIL[:, j] = out
		ads_silver_per_tank[:, j] = ads
		flow = out

	# Undissolved gold (tails) across 11 tanks
	r_tail = np.clip(0.72 + 0.20*(1 - O) + 0.12*robust_unit(ultraf), 0.65, 0.95)
	weights_tails = np.array([r_tail**k for k in range(2 + \
										S["tanks"]["cil"]["count"])]).T
	upset = np.random.default_rng(seed+800).random(n) < S["outliers"]["upset_day_rate"]
	if upset.any():
		bump = np.random.default_rng(seed+801).uniform(S["outliers"]["late_tank_bias_low"],
										S["outliers"]["late_tank_bias_high"], upset.sum())
		weights_tails[upset, -3:] *= (1 + bump[:, None])
	weights_tails = weights_tails / weights_tails.sum(axis=1, keepdims=True)
	Undiss_by_tank = Au_tails_g[:, None] * weights_tails
	Undiss_Leach = Undiss_by_tank[:, :2]
	Undiss_CIL = Undiss_by_tank[:, 2:]

	# Mass-balance rounding fixers
	def fix_sum(row, target):
		s = row.sum()
		if s == 0: return row
		diff = target - s
		row[np.argmax(row)] += diff
		return row
	for i in range(n):
		Undiss_by_tank[i, :] = fix_sum(Undiss_by_tank[i, :], Au_tails_g[i])
		ads_gold_per_tank[i, :] = fix_sum(ads_gold_per_tank[i, :], Au_ads_target[i])

	# Carbon loadings consistent with adsorption & counter-current
	V_cil = S["tanks"]["cil"]["volume_m3"]
	carbon_inventory_t = (carbon_cil * V_cil * 1000.0) / 1e6   # t
	RT = S["carbon"]["cil_carbon_residence_days_per_tank"]
	carbon_transfer_tpd = carbon_inventory_t / max(RT, 0.1)
	barren = S["carbon"]["barren_loading_gt"]
	load_floor = S["carbon"]["loading_floor_gt"]
	load_cap = S["carbon"]["loading_cap_gt"]

	Gold_Loadings = np.zeros_like(carbon_cil)
	for i in range(n):
		F = np.maximum(carbon_transfer_tpd[i, :], 1e-6)
		dL = ads_gold_per_tank[i, :] / F
		L = np.zeros(S["tanks"]["cil"]["count"] + 1)
		L[-1] = barren
		for j in range(S["tanks"]["cil"]["count"]-1, -1, -1):
			L[j] = L[j+1] + dL[j]
		if (L[0] > load_cap) or (L[-2] < load_floor):
			scale = 1.0
			if L[0] > 0: scale = min(scale, load_cap / L[0])
			if L[-2] > 0: scale = min(scale, max(1e-3, (load_floor - barren) / (L[-2] - barren)))
			dL *= scale
			L = np.zeros(S["tanks"]["cil"]["count"] + 1); L[-1] = barren
			for j in range(S["tanks"]["cil"]["count"]-1, -1, -1):
				L[j] = L[j+1] + dL[j]
			L = np.clip(L, load_floor, load_cap, out=L); L[-1] = barren
		Gold_Loadings[i, :] = L[:-1]
	Silver_Loadings = np.clip(0.20 * Gold_Loadings + np.random.default_rng(seed + 900).\
		normal(0, 5, Gold_Loadings.shape), 40, 300)
	
	# Averages across tanks (we’ll reuse them later and output as fields)
	# Free CN averages
	avg_free_cn_leach = 0.5 * (free_cn_tk1 + free_cn_tk2)
	avg_free_cn_cil   = cn_cil.mean(axis=1)
	avg_free_cn_tank  = (2*avg_free_cn_leach + ncil*avg_free_cn_cil) / (2 + ncil)

	# DO averages
	avg_do_leach = 0.5 * (do_leach_1 + do_leach_2)
	avg_do_cil   = do_cil.mean(axis=1)
	avg_do_tank  = (2*avg_do_leach + ncil*avg_do_cil) / (2 + ncil)

	# pH averages
	avg_ph_leach = 0.5 * (ph_leach_01 + ph_leach_02)
	avg_ph_cil   = ph_cil_all.mean(axis=1)
	avg_ph_tank  = (2*avg_ph_leach + ncil*avg_ph_cil) / (2 + ncil)

	# Carbon, loadings (averages across CIL)
	avg_carbon_gpl     = carbon_cil.mean(axis=1)
	avg_au_loading_gpl = Gold_Loadings.mean(axis=1)
	avg_ag_loading_gpl = Silver_Loadings.mean(axis=1)

	# Dissolved Au averages (solution concentration/proxy per tank position)
	au_diss_all = np.column_stack([
		Sol_Au_Leach_1_out,
		Sol_Au_Leach_2_out,
		Diss_Au_CIL
	])
	avg_au_diss_tank   = au_diss_all.mean(axis=1)
	avg_au_diss_leach  = np.column_stack([Sol_Au_Leach_1_out, Sol_Au_Leach_2_out]).mean(axis=1)
	avg_au_diss_cil    = Diss_Au_CIL.mean(axis=1)

	# CN mass in solution from average free CN (ppm ≡ mg/L)
	CN_in_solution_kg_day = (avg_free_cn_tank * L_per_day) / 1e6

	# ====================================================
	# Residence time (overall) in hours
	# ====================================================
	total_volume_m3 = S["tanks"]["leach"]["volume_m3"] * 2 + S["tanks"]["cil"]["volume_m3"] * ncil
	Residence_Time_hr = (total_volume_m3 / np.maximum(m3day, 1e-6)) * 24.0

	# ====================================================
	# P80 and weighted particle size
	# ====================================================
	# Use the helper you provided; note our column names are capitalised
	def _calc_p80_row(row):
		return calculate_p80(row, coarse_col="Coarse", fines_col="Fines", ultrafines_col="Ultrafines")

	def _calc_particle_size_row(row):
		return calculate_particle_size(row["Ultrafines"], row["Fines"], row["Coarse"],
									size_ultrafine=ultra_fine_sizing,
									size_fine=fine_sizing,
									size_coarse=coarse_sizing)

	# ====================================================
	# Interaction terms
	# ====================================================
	# Feed grade aliases (ppm): numerically identical to g/t day fields
	Au_Feed_Grade_ppm = au_day
	Ag_Feed_Grade_ppm = ag_day

	CN_to_Au_Ratio         = np.divide(NaCN_Used_Est_kgt, np.maximum(Au_Feed_Grade_ppm, 1e-9))
	CN_x_DO                = avg_free_cn_tank * avg_do_tank
	DO_Carbon_Interaction  = avg_do_cil * avg_carbon_gpl

	# Grade×Ultrafines, Throughput×Ultrafines, Grade×P80, Grade×RT
	Grade_x_Ultrafines     = au_day * ultraf
	Throughput_x_Ultrafines= m3day * ultraf
	# (P80 will be added after df is assembled, so we can multiply safely once present)
	# Grade_x_RT will use Residence_Time_hr which we already computed

	# =========================
	# Assemble output dataframe
	# =========================
	df = pd.DataFrame({
		"Date": dates,
		"Leach_Feed_Throughput_M3Hr":  np.round(m3hr_set, 3),
		"Leach_Feed_Throughput_M3Day": np.round(m3day, 3),
		"Percent_Solids": np.round(perc_solids, 3),

		"Leach_Feed_Grade_Au_Gt_Ds": np.round(au_ds, 3),
		"Leach_Feed_Grade_Au_Gt_Ns": np.round(au_ns, 3),
		"Leach_Feed_Grade_Au_Gt_Day": np.round(au_day, 3),

		"Leach_Feed_Grade_Ag_Gt_Ds": np.round(ag_ds, 3),
		"Leach_Feed_Grade_Ag_Gt_Ns": np.round(ag_ns, 3),
		"Leach_Feed_Grade_Ag_Gt_Day": np.round(ag_day, 3),

		"Coarse": np.round(coarse, 2),          # %
		"Fines":  np.round(fines, 2),           # %
		"Ultrafines": np.round(ultraf, 2),      # %

		"Free_CN_Leach_Tank_1_ppm": np.round(free_cn_tk1, 1),
		"Free_CN_Leach_Tank_2_ppm": np.round(free_cn_tk2, 1),

		"NaCN_Withdrawn_daily_kgpt": np.round(daily_cn_kgt, 4),
		"NaCN_Withdrawn_mtd_kgpt": np.round(mtd_cn_kgt, 4),
		"NaCN_Withdrawn_tpd": np.round((daily_cn_kgt * daily_tons) / 1000.0, 3),

		# Calculated, per your instruction:
		"Au_Leach_Feed_g": np.round(au_day * daily_tons, 1),

		"WAD_CN_Tailings_ppm": np.round(cn_tails_ppm, 0),

		# Daily tonnes:
		"Leach_Feed_Dry_t": np.round(daily_tons, 2),

		"DO_Leach_Tank_1_ppm": np.round(do_leach_1, 1),
		"DO_Leach_Tank_2_ppm": np.round(do_leach_2, 1),

		"Ph_Leach_Tank_01": np.round(ph_leach_01, 2),
		"Ph_Leach_Tank_02": np.round(ph_leach_02, 2),

		# Output tails (mass-balanced, includes small losses & inventory)
		"Au_Tailings_g": np.round(Au_tails_g, 1),

		# Recovered gold
		"Au_Recovered_g": np.round(Au_Recovered_g, 1),
    	"Recovery_pct":   np.round(Recovery_pct_out, 2),

		# --- Mass/solution/CN totals ---
		"Solids_Fraction":        np.round(Solids_Fraction, 5),
		"Liquid_Fraction":        np.round(Liquid_Fraction, 5),
		"Slurry_Mass_kg":         np.round(Slurry_Mass_kg, 1),
		"Solution_Mass_kg":       np.round(Solution_Mass_kg, 1),
		"CN_in_solution_kg_day":  np.round(CN_in_solution_kg_day, 3),
		"NaCN_Used_Est_kg_day":   np.round(NaCN_Used_Est_kg_day, 3),
		"NaCN_Used_Est_kgt":      np.round(NaCN_Used_Est_kgt, 5),

		# --- Residence time ---
		"Residence_Time_hr": np.round(Residence_Time_hr, 3),

		# --- Feed-grade ppm aliases (numerically equal to g/t) ---
		"Au_Feed_Grade_ppm": np.round(Au_Feed_Grade_ppm, 3),
		"Ag_Feed_Grade_ppm": np.round(Ag_Feed_Grade_ppm, 3),
	})

	# CIL CN/DO/Carbon/Loadings
	cil_cols = {}
	for i in range(S["tanks"]["cil"]["count"]):
		idx = i + 1
		cil_cols[f"Free_CN_Cil_Tank_{idx:02d}_ppm"] = np.round(cn_cil[:, i], 1)
		cil_cols[f"DO_Cil_Tank_{idx:02d}_ppm"] = np.round(do_cil[:, i], 1)
		cil_cols[f"Carbon_Concentration_Cil_Tank_{idx:02d}_gpl"] = np.round(carbon_cil[:, i], 1)
		cil_cols[f"Au_Loading_Cil_Tank_{idx:02d}_gptc"] = np.round(Gold_Loadings[:, i], 1)
		cil_cols[f"Ag_Loading_Cil_Tank_{idx:02d}_gptc"] = np.round(Silver_Loadings[:, i], 1)
		cil_cols[f"Ph_Cil_Tank_{idx:02d}"] = np.round(ph_cil_all[:, i], 2)

	df = pd.concat([df, pd.DataFrame(cil_cols)], axis=1)

	# Dissolved metal profiles
	df["Au_Dissolved_Leach_Tank_01_gpt"] = np.round(Sol_Au_Leach_1_out, 2)
	df["Au_Dissolved_Leach_Tank_02_gpt"] = np.round(Sol_Au_Leach_2_out, 2)
	for i in range(S["tanks"]["cil"]["count"]):
		df[f"Au_Dissolved_Cil_Tank_{i+1:02d}_gpt"] = np.round(Diss_Au_CIL[:, i], 2)
	df["Incremental_Au_Dissolved_Leach_Tank_01_g"] = np.round(Au_diss_leach_1, 2)
	df["Incremental_Au_Dissolved_Leach_Tank_02_g"] = np.round(Au_diss_leach_2, 2)

	df["Ag_Dissolved_Leach_Tank_01_gpt"] = np.round(Sol_Ag_Leach_1_out, 2)
	df["Ag_Dissolved_Leach_Tank_02_gpt"] = np.round(Sol_Ag_Leach_2_out, 2)
	for i in range(S["tanks"]["cil"]["count"]):
		df[f"Ag_Dissolved_Cil_Tank_{i+1:02d}_gpt"] = np.round(Diss_Ag_CIL[:, i], 2)
	df["Incremental_Ag_Dissolved_Leach_Tank_01_g"] = np.round(Ag_diss_leach_1, 2)
	df["Incremental_Ag_Dissolved_Leach_Tank_02_g"] = np.round(Ag_diss_leach_2, 2)

	# Undissolved gold per tank
	df["Au_Undissolved_Leach_Tank_01_gpdpt"] = np.round(Undiss_Leach[:, 0], 2)
	df["Au_Undissolved_Leach_Tank_02_gpdpt"] = np.round(Undiss_Leach[:, 1], 2)
	for i in range(S["tanks"]["cil"]["count"]):
		df[f"Au_Undissolved_Cil_Tank_{i+1:02d}_gpdpt"] = np.round(Undiss_CIL[:, i], 2)

	# =========================
	# Aggregations (requested)
	# =========================
	df["Avg_Free_CN_Tank_ppm"]  = np.round(avg_free_cn_tank, 3)
	df["Avg_Free_CN_Leach_ppm"] = np.round(avg_free_cn_leach, 3)
	df["Avg_Free_CN_Cil_ppm"]   = np.round(avg_free_cn_cil, 3)

	df["Avg_DO_Tank_ppm"]   = np.round(avg_do_tank, 3)
	df["Avg_DO_Leach_ppm"]  = np.round(avg_do_leach, 3)
	df["Avg_DO_Cil_ppm"]    = np.round(avg_do_cil, 3)

	df["Avg_Ph_Tank"]       = np.round(avg_ph_tank, 3)
	df["Avg_Ph_Leach_ppm"]  = np.round(avg_ph_leach, 3)
	df["Avg_Ph_Cil_ppm"]    = np.round(avg_ph_cil, 3)

	df["Avg_Carbon_Tank_gpl"]   = np.round(avg_carbon_gpl, 3)
	df["Avg_Au_Loading_Tank_gpl"] = np.round(avg_au_loading_gpl, 3)
	df["Avg_Ag_Loading_Tank_gpl"] = np.round(avg_ag_loading_gpl, 3)

	df["Avg_Au_Dissolved_Tank"]  = np.round(avg_au_diss_tank, 3)
	df["Avg_Au_Dissolved_Leach"] = np.round(avg_au_diss_leach, 3)
	df["Avg_Au_Dissolved_Cil"]   = np.round(avg_au_diss_cil, 3)

	# =========================
	# P80 & Particle Size
	# =========================
	df["P80_um"] = df.apply(_calc_p80_row, axis=1)
	df["Particle_Size"] = df.apply(_calc_particle_size_row, axis=1)

	# =========================
	# Interaction terms (use P80 now that it exists)
	# =========================
	df["Grade_x_Ultrafines"]       = np.round(Grade_x_Ultrafines, 6)
	df["Throughput_x_Ultrafines"]  = np.round(Throughput_x_Ultrafines, 6)
	df["Grade_x_P80"]              = np.round(df["Leach_Feed_Grade_Au_Gt_Day"] * df["P80_um"], 6)
	df["CN_to_Au_Ratio"]           = np.round(CN_to_Au_Ratio, 6)
	df["DO_Carbon_Interaction"]    = np.round(DO_Carbon_Interaction, 6)
	df["Grade_x_RT"]               = np.round(df["Leach_Feed_Grade_Au_Gt_Day"] * df["Residence_Time_hr"], 6)
	df["CN_x_DO"]                  = np.round(CN_x_DO, 6)

	return df

# -----------------------------
# Run code
# -----------------------------

df_demo = generate_demo_historical_full(
	start_date="2024-01-01",
	end_date="2025-12-31",
	seed=1337,
	settings=DEFAULT_SETTINGS
)
display(df_demo.head())


,Date,Leach_Feed_Throughput_M3Hr,Leach_Feed_Throughput_M3Day,Percent_Solids,Leach_Feed_Grade_Au_Gt_Ds,Leach_Feed_Grade_Au_Gt_Ns,Leach_Feed_Grade_Au_Gt_Day,Leach_Feed_Grade_Ag_Gt_Ds,Leach_Feed_Grade_Ag_Gt_Ns,Leach_Feed_Grade_Ag_Gt_Day,...,Avg_Au_Dissolved_Cil,P80_um,Particle_Size,Grade_x_Ultrafines,Throughput_x_Ultrafines,Grade_x_P80,CN_to_Au_Ratio,DO_Carbon_Interaction,Grade_x_RT,CN_x_DO
0,2024-01-01,144.033,2975.299,50.445,2.150,2.213,2.148,5.028,4.939,4.984,...,1758.258,77.795128,73.89375,156.555947,216853.730641,167.103935,0.479687,219.727369,73.637736,3417.443039
1,2024-01-02,143.967,3069.613,50.765,1.904,1.962,1.896,3.795,3.818,3.806,...,1631.811,81.570102,75.08250,136.750309,221356.418582,154.656914,0.571690,216.668871,63.002184,3832.628733
2,2024-01-03,145.731,3079.482,51.043,2.066,1.918,1.943,4.876,3.741,4.309,...,1773.748,78.684841,74.14375,140.567513,222767.292234,152.884646,0.519845,210.957578,64.356046,3119.132788
3,2024-01-04,147.231,2898.158,50.794,1.782,2.241,1.963,4.230,5.263,4.746,...,1535.622,73.343784,72.11750,143.568182,211994.499118,143.973848,0.528073,222.502137,69.087785,3830.531437
4,2024-01-05,146.811,2472.447,50.731,2.083,2.199,2.095,4.869,4.455,4.662,...,1501.355,78.248800,74.00000,151.843109,179190.104639,163.931235,0.503440,215.675751,86.429225,3472.485775


In [ ]:
df_demo.columns.to_list()

In [ ]:
# Load Shanta dataset
df_shanta = pd.read_csv("historical_shanta.csv")
df_shanta.columns.to_list()

In [ ]:
['Date',
 'Leach_Feed_Throughput_M3Hr',
 'Leach_Feed_Throughput_M3Day',
 'Percent_Solids',
 'Leach_Feed_Grade_Au_Gt_Ds',
 'Leach_Feed_Grade_Au_Gt_Ns',
 'Leach_Feed_Grade_Au_Gt_Day',
 'Leach_Feed_Grade_Ag_Gt_Ds',
 'Leach_Feed_Grade_Ag_Gt_Ns',
 'Leach_Feed_Grade_Ag_Gt_Day',
 'Free_CN_Leach_Tank_1_ppm',
 'Free_CN_Leach_Tank_2_ppm',
 'NaCN_Withdrawn_daily_kgpt',
 'NaCN_Withdrawn_mtd_kgpt',
 'NaCN_Withdrawn_tpd',
 'Au_Leach_Feed_g',
 'WAD_CN_Tailings_ppm',
 'Au_Tailings_g',
 'Leach_Feed_Gt_150um_Day',
 'Leach_Feed_Gt_75um_Day',
 'Leach_Feed_Lt_75um_Day',
 'Cyanide_Profile_Ppm_Leach_Tank_1',
 'Cyanide_Profile_Ppm_Leach_Tank_2',
 'Free_CN_Cil_Tank_1_ppm',
 'Free_CN_Cil_Tank_2_ppm',
 'Free_CN_Cil_Tank_3_ppm',
 'Free_CN_Cil_Tank_4_ppm',
 'Free_CN_Cil_Tank_5_ppm',
 'Free_CN_Cil_Tank_6_ppm',
 'Free_CN_Cil_Tank_7_ppm',
 'Free_CN_Cil_Tank_8_ppm',
 'Free_CN_Cil_Tank_9_ppm',
 'DO_Leach_Tank_1_ppm',
 'DO_Leach_Tank_2_ppm',
 'DO_Cil_Tank_1_ppm',
 'DO_Cil_Tank_2_ppm',
 'DO_Cil_Tank_3_ppm',
 'DO_Cil_Tank_4_ppm',
 'DO_Cil_Tank_5_ppm',
 'DO_Cil_Tank_6_ppm',
 'DO_Cil_Tank_7_ppm',
 'DO_Cil_Tank_8_ppm',
 'DO_Cil_Tank_9_ppm',
 'Carbon_Concentration_Cil_Tank_1_gpl',
 'Carbon_Concentration_Cil_Tank_2_gpl',
 'Carbon_Concentration_Cil_Tank_3_gpl',
 'Carbon_Concentration_Cil_Tank_4_gpl',
 'Carbon_Concentration_Cil_Tank_5_gpl',
 'Carbon_Concentration_Cil_Tank_6_gpl',
 'Carbon_Concentration_Cil_Tank_7_gpl',
 'Carbon_Concentration_Cil_Tank_8_gpl',
 'Carbon_Concentration_Cil_Tank_9_gpl',
 'Au_Loading_Cil_Tank_1_gptc',
 'Au_Loading_Cil_Tank_2_gptc',
 'Au_Loading_Cil_Tank_3_gptc',
 'Au_Loading_Cil_Tank_4_gptc',
 'Au_Loading_Cil_Tank_5_gptc',
 'Au_Loading_Cil_Tank_6_gptc',
 'Au_Loading_Cil_Tank_7_gptc',
 'Au_Loading_Cil_Tank_8_gptc',
 'Au_Loading_Cil_Tank_9_gptc',
 'Ag_Loading_Cil_Tank_1_gptc',
 'Ag_Loading_Cil_Tank_2_gptc',
 'Ag_Loading_Cil_Tank_3_gptc',
 'Ag_Loading_Cil_Tank_4_gptc',
 'Ag_Loading_Cil_Tank_5_gptc',
 'Ag_Loading_Cil_Tank_6_gptc',
 'Ag_Loading_Cil_Tank_7_gptc',
 'Ag_Loading_Cil_Tank_8_gptc',
 'Ag_Loading_Cil_Tank_9_gptc',
 'Au_Dissolved_Leach_Tank_1_gpt',
 'Au_Dissolved_Leach_Tank_2_gpt',
 'Au_Dissolved_Cil_Tank_1_gpt',
 'Au_Dissolved_Cil_Tank_2_gpt',
 'Au_Dissolved_Cil_Tank_3_gpt',
 'Au_Dissolved_Cil_Tank_4_gpt',
 'Au_Dissolved_Cil_Tank_5_gpt',
 'Au_Dissolved_Cil_Tank_6_gpt',
 'Au_Dissolved_Cil_Tank_7_gpt',
 'Au_Dissolved_Cil_Tank_8_gpt',
 'Au_Dissolved_Cil_Tank_9_gpt',
 'Ag_Dissolved_Leach_Tank_1_gpt',
 'Ag_Dissolved_Leach_Tank_2_gpt',
 'Ag_Dissolved_Cil_Tank_1_gpt',
 'Ag_Dissolved_Cil_Tank_2_gpt',
 'Ag_Dissolved_Cil_Tank_3_gpt',
 'Ag_Dissolved_Cil_Tank_4_gpt',
 'Ag_Dissolved_Cil_Tank_5_gpt',
 'Ag_Dissolved_Cil_Tank_6_gpt',
 'Ag_Dissolved_Cil_Tank_7_gpt',
 'Ag_Dissolved_Cil_Tank_8_gpt',
 'Au_Undissolved_Leach_Tank_1_gpdpt',
 'Au_Undissolved_Leach_Tank_2_gpdpt',
 'Au_Undissolved_Cil_Tank_1_gpdpt',
 'Au_Undissolved_Cil_Tank_2_gpdpt',
 'Au_Undissolved_Cil_Tank_3_gpdpt',
 'Au_Undissolved_Cil_Tank_4_gpdpt',
 'Au_Undissolved_Cil_Tank_5_gpdpt',
 'Au_Undissolved_Cil_Tank_6_gpdpt',
 'Au_Undissolved_Cil_Tank_7_gpdpt',
 'Au_Undissolved_Cil_Tank_8_gpdpt',
 'Au_Undissolved_Cil_Tank_9_gpdpt',
 'Leach_Feed_Dry_t',
 'Ph_Leach_Tank_01',
 'Ph_Leach_Tank_02',
 'Ph_Cil_Tank_01',
 'Ph_Cil_Tank_02',
 'Ph_Cil_Tank_03',
 'Ph_Cil_Tank_04',
 'Ph_Cil_Tank_05',
 'Ph_Cil_Tank_06',
 'Ph_Cil_Tank_07',
 'Ph_Cil_Tank_08',
 'Ph_Cil_Tank_09',
 'Au_Dissolved_Leach_Tank_1_g',
 'Au_Dissolved_Leach_Tank_2_g',
 'Au_Dissolved_Cil_Tank_1_g',
 'Au_Dissolved_Cil_Tank_2_g',
 'Au_Dissolved_Cil_Tank_3_g',
 'Au_Dissolved_Cil_Tank_4_g',
 'Au_Dissolved_Cil_Tank_5_g',
 'Au_Dissolved_Cil_Tank_6_g',
 'Au_Dissolved_Cil_Tank_7_g',
 'Au_Dissolved_Cil_Tank_8_g',
 'Au_Dissolved_Cil_Tank_9_g']

In [16]:
df_demo.to_csv("historical_demo.csv", index=False)

In [ ]:
feature_names = [
 'Leach_Feed_Throughput_M3Hr',
 'Leach_Feed_Throughput_M3Day',
 'Percent_Solids',
 'Leach_Feed_Grade_Au_Gt_Day',
 'Leach_Feed_Grade_Ag_Gt_Day',
 'Coarse',
 'Fines',
 'Ultrafines',]
#  'Free_CN_Leach_Tank_1_ppm',
#  'Free_CN_Leach_Tank_2_ppm',
#  'Daily_Nacn_Consumption_Kgt',
#  'Mtd_Nacn_Consumption_Kgt',
#  'Nacn_Used_T',
#  'Au_Leach_Feed_g',
#  'WAD_CN_Tailings_ppm',
#  'Leach_Feed_Dry_t',
#  'DO_Leach_Tank_1_ppm',
#  'DO_Leach_Tank_2_ppm',
#  'Ph_Leach_Tank_01',
#  'Ph_Leach_Tank_02',
#  'Au_Tailings_g',
#  'Cyanide_Profile_Ppm_Cil_Tank_01',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_01',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_01',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_01',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_01',
#  'Cyanide_Profile_Ppm_Cil_Tank_02',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_02',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_02',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_02',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_02',
#  'Cyanide_Profile_Ppm_Cil_Tank_03',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_03',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_03',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_03',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_03',
#  'Cyanide_Profile_Ppm_Cil_Tank_04',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_04',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_04',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_04',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_04',
#  'Cyanide_Profile_Ppm_Cil_Tank_05',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_05',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_05',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_05',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_05',
#  'Cyanide_Profile_Ppm_Cil_Tank_06',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_06',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_06',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_06',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_06',
#  'Cyanide_Profile_Ppm_Cil_Tank_07',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_07',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_07',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_07',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_07',
#  'Cyanide_Profile_Ppm_Cil_Tank_08',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_08',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_08',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_08',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_08',
#  'Cyanide_Profile_Ppm_Cil_Tank_09',
#  'Dissolved_Oxygen_Profile_Ppm_Cil_Tank_09',
#  'Carbon_Concentration_Profile_Gl_Cil_Tank_09',
#  'Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_09',
#  'Silver_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_09',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Leach_Tank_01',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Leach_Tank_02',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_01',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_02',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_03',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_04',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_05',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_06',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_07',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_08',
#  'Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_09',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Leach_Tank_01',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Leach_Tank_02',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_01',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_02',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_03',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_04',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_05',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_06',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_07',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_08',
#  'Dissolved_Silver_Metal_Profile_Gmilled_Tons_Cil_Tank_09',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Leach_Tank_01',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Leach_Tank_02',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_01',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_02',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_03',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_04',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_05',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_06',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_07',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_08',
#  'Undissolved_Gold_Profile_Au_Grams_Per_Day_Per_Tank_Cil_Tank_09']

In [ ]:
# Generate a series of plots for our new dataset
# First show series of historigram subplots for each feature

import matplotlib.pyplot as plt
from plotting_functions_v2 import (
	plot_bar, plot_scatter, plot_scatter_with_regression, plot_histogram,
	plot_response_surface
)

# Define new KDA based histogram plotting function
def plot_kda_histograms(data, feature_names):
	n_features = len(feature_names)
	fig, axes = plt.subplots(n_features, 1, figsize=(10, 5 * n_features))
	for i, col in enumerate(feature_names):
		axes[i].hist(data[col], bins=30, color='green', alpha=0.7)
		axes[i].set_title(f"KDA Histogram of {col}")
		axes[i].set_xlabel(col)
		axes[i].set_ylabel("Frequency")
	plt.tight_layout()
	plt.show()


# Plot histograms for each feature
for feat in feature_names:
	plot_histogram(
		data=df_demo,
		column=feat,
		title=f'Distribution of {feat}',
		vline=df_demo[feat].mean(),
		vline_label='Mean',
		xlabel=feat,
		bins=30,
		color='blue',
		kde=True
	)


In [ ]:
# =========================
# Comparative plots for demo dataset
# =========================
import numpy as np
import matplotlib.pyplot as plt

# ---- helpers ----
def robust_unit(x):
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanpercentile(x, [5, 95])
    if hi - lo < 1e-9:
        return np.zeros_like(x)
    return np.clip((x - lo) / (hi - lo), 0, 1)

def compute_grind_index(df, settings):
    sizes = settings["grind"]["particle_sizes_um"]
    surface_index = (
        df["Coarse"]      / sizes["Coarse"] +
        df["Fines"]       / sizes["Fines"] +
        df["Ultrafines"]  / sizes["Ultrafines"]
    )
    return robust_unit(surface_index)

def scatter_with_bins(ax, x, y, xlab, ylab, title, nbins=12):
    ax.scatter(x, y, s=10, alpha=0.5)
    # bin-mean line (no smoothing libs)
    qs = np.linspace(0, 1, nbins+1)
    edges = np.quantile(x, qs)
    mids, means = [], []
    for i in range(nbins):
        m = (x >= edges[i]) & (x <= edges[i+1])
        if m.sum() >= 5:
            mids.append(0.5*(edges[i] + edges[i+1]))
            means.append(np.nanmean(y[m]))
    if len(mids) > 1:
        ax.plot(mids, means, linewidth=2)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.set_title(title)
    ax.grid(False)

# ---- derived features ----
df = df_demo.copy()

# Estimated recovery (if you didn’t store bullion explicitly):
df["Recovery_Est_Pct"] = (100.0 * (1.0 - df["Au_Tailings_g"] / df["Au_Leach_Feed_g"])).clip(70, 98)

# Mean leach DO
df["DO_Leach_Mean"] = 0.5*(df["DO_Leach_Tank_1_ppm"] +
                           df["DO_Leach_Tank_2_ppm"])

# Grind index (responds to Coarse/Fines/Ultrafines + sizes in settings)
df["Grind_Index"] = compute_grind_index(df, S)

# Runtime / downtime from volumetric balance
df["Runtime_Hours"]  = df["Leach_Feed_Throughput_M3Day"] / df["Leach_Feed_Throughput_M3Hr"]
df["Downtime_Hours"] = 24.0 - df["Runtime_Hours"]

# Convenience handles
cn_kgt   = df["NaCN_Withdrawn_tpd"].values
rec_pct  = df["Recovery_Est_Pct"].values
do_mean  = df["DO_Leach_Mean"].values
grind    = df["Grind_Index"].values

# =========================
# 1) Recovery vs CN dosage
# =========================
fig, ax = plt.subplots(figsize=(7,5))
scatter_with_bins(ax, cn_kgt, rec_pct, "CN consumption (kg/t)", "Recovery (%)",
                  "Recovery vs Cyanide Dosage")

# =========================
# 2) Recovery vs DO (leach)
# =========================
fig, ax = plt.subplots(figsize=(7,5))
scatter_with_bins(ax, do_mean, rec_pct, "Mean DO in leach (ppm)", "Recovery (%)",
                  "Recovery vs Dissolved Oxygen (Leach)")

# =========================
# 3) Recovery vs Grind Index
# =========================
fig, ax = plt.subplots(figsize=(7,5))
scatter_with_bins(ax, grind, rec_pct, "Grind Reactivity Index (0–1)", "Recovery (%)",
                  "Recovery vs Grind Reactivity")

# =========================
# 4) Tailings Au vs CN in tails
# =========================
fig, ax = plt.subplots(figsize=(7,5))
scatter_with_bins(ax, df["WAD_CN_Tailings_ppm"].values,
                  df["Au_Tailings_g"].values,
                  "CN in tails (ppm)", "Au in tails (g)",
                  "Tailings Au vs CN in Tails")

# =========================
# 5) Ag vs Au grade (daily)
# =========================
fig, ax = plt.subplots(figsize=(7,5))
x = df["Leach_Feed_Grade_Au_Gt_Day"].values
y = df["Leach_Feed_Grade_Ag_Gt_Day"].values
ax.scatter(x, y, s=10, alpha=0.5)
# Reference bands (Ag = 1.9–2.4 × Au)
xx = np.linspace(np.nanmin(x), np.nanmax(x), 50)
ax.plot(xx, 1.9*xx, linewidth=2)
ax.plot(xx, 2.4*xx, linewidth=2)
ax.set_xlabel("Au grade (g/t)"); ax.set_ylabel("Ag grade (g/t)")
ax.set_title("Ag vs Au Grade (with 1.9× and 2.4× bands)")
ax.grid(False)

# =========================
# 6) Throughput setpoint vs runtime
# =========================
fig, ax = plt.subplots(figsize=(7,5))
scatter_with_bins(ax, df["Leach_Feed_Throughput_M3Hr"].values,
                  df["Runtime_Hours"].values,
                  "Throughput (m³/h)", "Runtime (h)",
                  "Throughput Setpoint vs Runtime")

# =========================
# 7) Time-series: Recovery and CN dosage
# =========================
fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(df["Date"], rec_pct, linewidth=1.5)
ax1.set_ylabel("Recovery (%)"); ax1.set_xlabel("Date"); ax1.set_title("Recovery and CN Dosage Over Time")
ax1.grid(False)
ax2 = ax1.twinx()
ax2.plot(df["Date"], cn_kgt, linewidth=1.0)
ax2.set_ylabel("CN (kg/t)")
ax2.grid(False)

# =========================
# 8) Per-train profiles (one representative day)
# =========================
# pick a median-recovery day
mid_idx = int(np.argsort(rec_pct)[len(df)//2])

# Dissolved Au profile across tanks (Leach 1–2, then CIL 1–9)
leach_a = df.loc[mid_idx, "Dissolved_Gold_Metal_Profile_G_Milled_Tons_Leach_Tank_01"]
leach_b = df.loc[mid_idx, "Dissolved_Gold_Metal_Profile_G_Milled_Tons_Leach_Tank_02"]
cil_cols = [f"Dissolved_Gold_Metal_Profile_G_Milled_Tons_Cil_Tank_{k:02d}" for k in range(1,10)]
cil_vals = df.loc[mid_idx, cil_cols].values
prof = np.concatenate([[leach_a, leach_b], cil_vals])
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(np.arange(1, 11+1), prof, marker="o")
ax.set_xticks(np.arange(1, 12))
ax.set_xlabel("Tank number (2 Leach + 9 CIL)"); ax.set_ylabel("Dissolved Au (g)")
ax.set_title("Dissolved Gold Profile (median-recovery day)")
ax.grid(False)

# Carbon loadings profile same day
load_cols = [f"Gold_Carbon_Loadings_Profile_Gt_Of_Carbon_Cil_Tank_{k:02d}" for k in range(1,10)]
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(np.arange(1, 9+1), df.loc[mid_idx, load_cols].values, marker="o")
ax.set_xticks(np.arange(1, 10))
ax.set_xlabel("CIL tank number"); ax.set_ylabel("Gold loading (g/t C)")
ax.set_title("Carbon Loadings Profile (median-recovery day)")
ax.grid(False)

plt.tight_layout()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df = df_demo.copy()

# Recovery if not already present
if "Recovery_Est_Pct" not in df.columns:
    rec = 100.0*(1.0 - df["Au_Tailings_g"].to_numpy() /
                 np.maximum(df["Au_Leach_Feed_g"].to_numpy(), 1e-9))
    df["Recovery_Est_Pct"] = np.clip(rec, 70, 99.5)

# Features
x = df["Daily_Nacn_Consumption_Kgt"].to_numpy()  # CN (kg/t)
y = 0.5*(df["DO_Leach_Tank_1_ppm"].to_numpy() +
         df["DO_Leach_Tank_2_ppm"].to_numpy())  # DO (ppm)
c = df["Recovery_Est_Pct"].to_numpy()  # color = recovery

# Scatter
fig, ax = plt.subplots(figsize=(7,5))
sc = ax.scatter(x, y, c=c, s=14, alpha=0.65)
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Recovery (%)")
ax.set_xlabel("CN consumption (kg/t)")
ax.set_ylabel("Mean DO in leach (ppm)")
ax.set_title("CN vs DO (points colored by Recovery)")
ax.grid(False)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Bin edges (tweak as desired)
cn_bins = np.linspace(np.nanmin(x), np.nanmax(x), 26)  # 25 bins
do_bins = np.linspace(np.nanmin(y), np.nanmax(y), 26)  # 25 bins

# Aggregate mean recovery per 2D bin
sum_rec, _, _ = np.histogram2d(x, y, bins=[cn_bins, do_bins], weights=c)
count,   _, _ = np.histogram2d(x, y, bins=[cn_bins, do_bins])
mean_rec = sum_rec / np.maximum(count, 1)               # avoid /0
mean_rec[count == 0] = np.nan                           # mark empty bins

# Plot heatmap
X, Y = np.meshgrid(cn_bins, do_bins)                    # grid from bin edges
Z = np.ma.masked_invalid(mean_rec.T)                    # shape: (len(do_bins)-1, len(cn_bins)-1)

fig, ax = plt.subplots(figsize=(7,5))
pm = ax.pcolormesh(X, Y, Z, shading="auto")             # default colormap
cb = plt.colorbar(pm, ax=ax)
cb.set_label("Mean recovery (%)")
ax.set_xlabel("CN consumption (kg/t)")
ax.set_ylabel("Mean DO in leach (ppm)")
ax.set_title("Mean Recovery by CN × DO (binned heatmap)")
ax.grid(False)
plt.show()
